- 왼쪽 카메라 이미지 `*_left.png`
- 오른쪽 카메라 이미지 `*_right.png`
- 시차(disparity) 이미지 (`*_disp.png` 또는 `*_disp16.png`)
- confidnece 이미지 (`*_confidence.png` 등)

In [1]:
import os
import re
import math
import numpy as np
import pandas as pd
import cv2
from configparser import ConfigParser

# -----------------------------
# 1) conf 읽기
# -----------------------------
def load_conf(conf_path: str, section: str = "LEFT_CAM_FHD"):
    cp = ConfigParser()
    cp.read(conf_path, encoding="utf-8")
    fx = float(cp[section]["fx"])
    fy = float(cp[section]["fy"])
    cx = float(cp[section]["cx"])
    cy = float(cp[section]["cy"])
    baseline_mm = float(cp["STEREO"]["BaseLine"])
    return {"fx": fx, "fy": fy, "cx": cx, "cy": cy, "baseline_m": baseline_mm / 1000.0}

# -----------------------------
# 2) disp16 스케일 추정 (최소 규칙)
# -----------------------------
def normalize_disp16(disp16: np.ndarray):
    # disp16: uint16
    m = float(np.nanmax(disp16))
    if m > 2048:          # 값이 너무 크면 스케일이 들어갔다고 가정
        return disp16.astype(np.float32) / 16.0, 16.0
    else:
        return disp16.astype(np.float32), 1.0

# -----------------------------
# 3) disparity -> depth (Z)
# -----------------------------
def disparity_to_depth(disp: np.ndarray, fx: float, baseline_m: float):
    # disp in pixels, fx in pixels
    eps = 1e-6
    return (fx * baseline_m) / (disp + eps)

# -----------------------------
# 4) ROI에서 3D 포인트 생성
# -----------------------------
def points_from_depth(depth: np.ndarray, K: dict, mask: np.ndarray, roi_y0: int, stride: int = 6):
    H, W = depth.shape
    fx, fy, cx, cy = K["fx"], K["fy"], K["cx"], K["cy"]

    ys = np.arange(roi_y0, H, stride)
    xs = np.arange(0, W, stride)
    uu, vv = np.meshgrid(xs, ys)  # uu: x, vv: y

    m = mask[roi_y0:H:stride, 0:W:stride] > 0
    z = depth[roi_y0:H:stride, 0:W:stride][m]

    u = uu[m].astype(np.float32)
    v = vv[m].astype(np.float32)

    x = (u - cx) * z / fx
    y = (v - cy) * z / fy
    pts = np.stack([x, y, z], axis=1)  # (N,3)
    return pts

# -----------------------------
# 5) 평면 피팅 (RANSAC 최소 구현)
# plane: n·p + d = 0, n is unit
# -----------------------------
def fit_plane_ransac(points: np.ndarray, iters: int = 300, dist_th: float = 0.02):
    # dist_th: meters. (2cm 정도로 시작)
    if len(points) < 200:
        return None

    best_inliers = None
    best_model = None
    N = len(points)
    rng = np.random.default_rng(0)

    for _ in range(iters):
        idx = rng.choice(N, size=3, replace=False)
        p1, p2, p3 = points[idx]
        v1, v2 = p2 - p1, p3 - p1
        n = np.cross(v1, v2)
        norm = np.linalg.norm(n)
        if norm < 1e-6:
            continue
        n = n / norm
        d = -np.dot(n, p1)

        # point-to-plane distance
        dist = np.abs(points @ n + d)
        inliers = dist < dist_th

        if best_inliers is None or inliers.sum() > best_inliers.sum():
            best_inliers = inliers
            best_model = (n, d)

    if best_model is None:
        return None

    # inliers로 재추정(SVD)
    inlier_pts = points[best_inliers]
    c = inlier_pts.mean(axis=0)
    A = inlier_pts - c
    _, _, vh = np.linalg.svd(A, full_matrices=False)
    n = vh[-1]
    n = n / (np.linalg.norm(n) + 1e-8)
    d = -np.dot(n, c)

    return {"n": n, "d": d, "inlier_ratio": float(best_inliers.mean())}

# -----------------------------
# 6) 법선 -> slope (grade %)
# 카메라 좌표: x 오른쪽, y 아래, z 전방 가정
# 중력 위쪽을 g=(0,-1,0)로 둠 (카메라가 수평에 가깝다고 가정)
# -----------------------------
def normal_to_grade(n: np.ndarray):
    g = np.array([0.0, -1.0, 0.0], dtype=np.float32)  # up direction
    cosang = float(np.clip(abs(np.dot(n, g)), 0.0, 1.0))
    angle = math.acos(cosang)          # radians
    grade = math.tan(angle) * 100.0    # %
    return grade, angle * 180.0 / math.pi

# -----------------------------
# 7) 한 프레임 처리
# -----------------------------
# 한 프레임(=disp16 1장 + confidence 1장 + conf 1개)에서 slope(경사%)를 뽑아내는 메인 파이프라인
'''
입력:
disp16_path: disparity16 이미지 파일 경로
conf_bin_path: confidence(2진 마스크) 이미지 경로
conf_path: 카메라 파라미터 .conf 파일 경로

옵션:
roi_ratio=0.6: 이미지 높이의 60% 지점부터 아래쪽만 “지면 후보”로 사용
stride=6: 픽셀을 6칸마다 샘플링해서 속도/메모리 줄임
'''
def extract_slope_from_frame(disp16_path: str, conf_bin_path: str, conf_path: str,
                             roi_ratio: float = 0.6, stride: int = 6):
    K = load_conf(conf_path, section="LEFT_CAM_FHD") # conf 파일을 읽어서 카메라 파라미터를 가져옴, K 안에는 보통 이런 값이 들어있음: fx, fy, cx, cy (intrinsics) baseline_m (미터 단위 baseline)

    disp16 = cv2.imread(disp16_path, cv2.IMREAD_UNCHANGED) # disp16 이미지를 로드, IMREAD_UNCHANGED는 “원본 비트 깊이 유지” disp16은 보통 uint16이므로 이 옵션이 중요

    if disp16 is None: # 파일이 없거나 읽기 실패하면 즉시 에러 발생
        raise FileNotFoundError(disp16_path)
    if disp16.dtype != np.uint16: # 만약 uint8로 읽혔다면 depth 계산이 다 망가질 수 있음 → 강제 에러
        raise ValueError(f"disp16 must be uint16, got {disp16.dtype}")

    conf_bin = cv2.imread(conf_bin_path, cv2.IMREAD_GRAYSCALE) # confidence 이미지를 그레이스케일로 로드, 여기서는 “마스크(0/255)”라고 가정하므로 RGB가 필요 없음
    if conf_bin is None: # 파일이 없거나 읽기 실패하면 즉시 에러 발생
        raise FileNotFoundError(conf_bin_path)

    disp, scale = normalize_disp16(disp16) # disp16의 “스케일 문제”를 처리하는 함수 | 이 함수를 통해 disp: float32 disparity(px 추정치), scale: disp16을 몇으로 나눴는지(예: 16.0 or 1.0)를 기록


    '''
    “사용할 수 있는 픽셀”만 True로 만드는 마스크

    조건 2개:
    confidence가 0이 아니어야 함(신뢰되는 픽셀)
    disparity가 거의 0이 아니어야 함(0이면 depth가 무한대로 튐)
    1e-3은 0 근처를 피하기 위한 아주 작은 안전장치
    '''
    mask = (conf_bin > 0) & (disp > 1e-3)

    '''
    mask는 True/False 배열

    .mean()을 하면 True=1, False=0으로 계산되어
    전체 픽셀 중 True 비율(=유효 픽셀 비율)이 나옴
    예: 0.35면 35% 픽셀이 유효
    '''
    valid_ratio = float(mask.mean())

    '''
    confidence True 픽셀 비율이 20% 이상인 프레임만 사용
    20% 미만이면 이 프레임은 slope 계산을 하지 않고 “실패”로 반환

    raise가 아니라 return인 이유:
    폴더 전체 batch 처리할 때 “이 프레임만 스킵하고 계속”하기 위함
    '''
    if valid_ratio < 0.20:
        return {"ok": False, "reason": "low_confidence_frame", "valid_ratio": valid_ratio}

    '''
    disparity를 depth(거리)로 변환
    공식은 내부에서:
    Z = fx * baseline / disp
    출력 depth는 이미지와 같은 HxW 형태의 float 배열(미터 단위)
    '''
    depth = disparity_to_depth(disp, K["fx"], K["baseline_m"])

    H, W = depth.shape # depth 이미지의 높이/너비를 얻음
    roi_y0 = int(H * roi_ratio) # ROI 시작 y좌표 계산,예: H=592, roi_ratio=0.6 → roi_y0=355 즉, y=355부터 아래쪽만 “지면 후보”로 씀

    pts = points_from_depth(depth, K, mask.astype(np.uint8), roi_y0=roi_y0, stride=stride) # depth를 3D 점들로 바꿈. ROI(하단부) 영역에서 mask가 True인 픽셀만 stride 간격으로 샘플링해서 (X,Y,Z) 3D 점들을 N개 뽑음 결과: pts는 shape (N, 3)인 numpy 배열
    model = fit_plane_ransac(pts, iters=300, dist_th=0.03)  # 3D 점들에 “평면”을 맞추는 단계/ RANSAC을 쓰는 이유: 지면 외에도 사람/벽/차량 같은 이상치 점이 섞이기 때문 RANSAC은 이상치를 버리고 “가장 많은 점이 지지하는 평면”을 찾음

    if model is None or model["inlier_ratio"] < 0.30: # 평면 추정이 실패했거나, 평면에 속하는 점 비율이 너무 낮으면(=지면 평면이 제대로 안 잡힘) slope를 신뢰할 수 없으니 실패로 반환
        return {"ok": False, "reason": "plane_fit_failed", "valid_ratio": valid_ratio}

    grade, angle_deg = normal_to_grade(model["n"]) # 평면 법선 벡터 n을 이용해서 경사 계산

    return {
        "ok": True,
        "disp_scale_div": scale,
        "valid_ratio": valid_ratio,
        "inlier_ratio": model["inlier_ratio"],
        "grade_pct": grade,
        "slope_angle_deg": angle_deg,
        "normal": model["n"],
        "K": K,
    }

In [4]:
from glob import glob # 파일 경로를 패턴으로 검색해 리스트로 가져오는 함수

# depth_dir 폴더 안에서 *_disp16.png 파일을 전부 찾고, 각 disp16 파일과 같은 id를 가진 confidence 파일을 찾아서 (id, disp16_path, conf_path) 형태의 DataFrame을 만든다.
def build_frame_index(depth_dir: str):
    disp16_paths = sorted(glob(os.path.join(depth_dir, "*_disp16.png"))) # glob 함수를 이용해서 _disp16.png에 해당하는 파일 Path 리스트를 생성
    rows = [] # 
    for p in disp16_paths: # 파일 path를 반복
        base = re.sub(r"_disp16\.png$", "", os.path.basename(p)) # os.path.basename(p) → 경로에서 파일명만 추출, re.sub(...) → 끝의 "_disp16.png"를 제거해서 **공통 id(base)**를 만듦 결과: "ZED1_KSC_001032"
        cand = [ # confidence 파일 후보들 (둘 중 존재하는 걸 사용)
            os.path.join(depth_dir, base + "_confidence.png"),
            os.path.join(depth_dir, base + "_confidence_save.png"),  # 이건 binary가 아닐 수 있어도 일단 후보
        ]
        conf_path = None # 아직 어떤 confidence 파일을 쓸지 결정 못했으니 None으로 시작
        for c in cand: # 후보 리스트 cand를 순서대로 보면서
            if os.path.exists(c): # 파일이 실제로 존재하면(os.path.exists)
                conf_path = c # conf_path에 저장하고
                break # 루프 종료
        rows.append({"id": base, "disp16_path": p, "conf_path": conf_path}) # 이 프레임(=한 장)에 대한 정보를 딕셔너리로 만들어 rows에 추가, conf_path가 없으면 None이 들어감(나중에 처리)
    return pd.DataFrame(rows)

# build_frame_index로 “처리할 프레임 목록”을 만든 뒤, 각 프레임에 대해 extract_slope_from_frame()를 호출해 slope를 계산하고, 그 결과를 모아 DataFrame으로 반환한다.
def run_depth_folder(depth_dir: str, conf_file: str):
    df = build_frame_index(depth_dir) # 앞에서 만든 DataFrame 생성
    out = []
    for _, r in df.iterrows(): # DataFrame의 각 행을 순회 (r은 한 행(Series)이고, 그 안에 id, disp16_path, conf_path가 있음, _는 index인데 여기서는 필요 없어서 버리는 용도)
        if r["conf_path"] is None: # confidence 파일이 없으면 slope 계산이 불가능(또는 품질 필터 못 함)
            out.append({"id": r["id"], "ok": False, "reason": "no_confidence_file"})
            continue
        res = extract_slope_from_frame( # 핵심 처리 함수 호출
            disp16_path=r["disp16_path"], # disparity16 이미지 경로
            conf_bin_path=r["conf_path"], # confidence 마스크(또는 confidence 이미지) 경로
            conf_path=conf_file, # 카메라 파라미터(conf)
            roi_ratio=0.6, # 이미지 아래쪽 40%를 ROI로 사용(0.6~1.0)
            stride=6 # 6픽셀 간격으로 샘플링해서 연산량 줄임(속도)
        )
        out.append({"id": r["id"], **res}) # 결과를 out에 저장 (id를 먼저 넣고 res 딕셔너리의 키들을 펼쳐서(**) 함께 합침)
        # 예: res가 {"ok": True, "grade_pct": 9.2}면 최종은 {"id": "...", "ok": True, "grade_pct": 9.2}
    return pd.DataFrame(out)

# 입력 폴더와 conf 파일 경로 지정
depth_dir = "../dl_datas/Depth_002"
conf_file = "../dl_datas/Depth_002/Depth_002.conf"

# Depth_001 폴더 안의 프레임들을 싹 돌려서 결과 테이블 생성
result_df = run_depth_folder(depth_dir, conf_file)
result_df.sort_values(["ok","grade_pct"], ascending=[False, False]).head(10)

,id,ok,reason,valid_ratio,disp_scale_div,inlier_ratio,grade_pct,slope_angle_deg,normal,K
10,ZED1_KSC_001854,True,NaN,0.286774,16.0,0.824645,23.469505,13.207992,"[-0.0036727968, 0.97354704, -0.22845723]","{'fx': 1400.15, 'fy': 1400.15, 'cx': 943.093, ..."
11,ZED1_KSC_001855,True,NaN,0.289993,16.0,0.822500,22.316714,12.580389,"[-0.032154985, 0.97599137, -0.2154226]","{'fx': 1400.15, 'fy': 1400.15, 'cx': 943.093, ..."
18,ZED1_KSC_002351,True,NaN,0.207799,16.0,0.797359,21.320608,12.035601,"[0.004601859, -0.9780182, 0.20846882]","{'fx': 1400.15, 'fy': 1400.15, 'cx': 943.093, ..."
8,ZED1_KSC_001852,True,NaN,0.288946,16.0,0.885661,21.285649,12.016441,"[0.01106589, -0.9780879, 0.20789808]","{'fx': 1400.15, 'fy': 1400.15, 'cx': 943.093, ..."
7,ZED1_KSC_001851,True,NaN,0.276680,16.0,0.870324,21.195102,11.966801,"[-0.010661192, 0.9782679, -0.20707071]","{'fx': 1400.15, 'fy': 1400.15, 'cx': 943.093, ..."
12,ZED1_KSC_001856,True,NaN,0.287634,16.0,0.856607,20.331665,11.492537,"[-0.0050502215, 0.97995067, -0.19917648]","{'fx': 1400.15, 'fy': 1400.15, 'cx': 943.093, ..."
6,ZED1_KSC_001849,True,NaN,0.295291,16.0,0.810141,20.032495,11.327834,"[-0.0016375594, -0.98051935, 0.19641589]","{'fx': 1400.15, 'fy': 1400.15, 'cx': 943.093, ..."
14,ZED1_KSC_001862,True,NaN,0.300024,16.0,0.859012,15.972032,9.074652,"[0.0037391968, 0.9874837, -0.15767696]","{'fx': 1400.15, 'fy': 1400.15, 'cx': 943.093, ..."
9,ZED1_KSC_001853,True,NaN,0.279018,16.0,0.849075,12.888704,7.344195,"[0.011259388, 0.99179614, -0.12733318]","{'fx': 1400.15, 'fy': 1400.15, 'cx': 943.093, ..."
20,ZED1_KSC_002355,True,NaN,0.273085,16.0,0.839881,12.542454,7.148965,"[-0.0046432717, -0.99222594, 0.124362916]","{'fx': 1400.15, 'fy': 1400.15, 'cx': 943.093, ..."
